In [ ]:
!pip install dvc dvc-gdrive
!pip install wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 470.1/470.1 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.3/79.3 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 451.2/451.2 kB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.0/45.0 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.2/214.2 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.2/74.2 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 73.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.7/

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import librosa
import librosa.display

import wandb

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d awsaf49/asvpoof-2019-dataset

!unzip -q asvpoof-2019-dataset.zip -d /content/asvspoof_data

Dataset URL: https://www.kaggle.com/datasets/awsaf49/asvpoof-2019-dataset
License(s): ODC Attribution License (ODC-By)
100% 23.6G/23.6G [04:08<00:00, 102MB/s]



In [ ]:
import os
import pandas as pd
import torch
import torchaudio
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

class ASVSpoofDataset(Dataset):
    def __init__(self, protocol_file, audio_dir, transform=None, max_length=64000):
        """
        max_length: 64,000 samples equals exactly 4 seconds of audio at a 16kHz sample rate.
        """
        self.data = pd.read_csv(
            protocol_file, sep=' ', header=None,
            names=['Speaker_ID', 'Audio_File_Name', 'Environment', 'Attack', 'Label']
        )
        self.audio_dir = audio_dir
        self.transform = transform
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        audio_name = self.data.iloc[idx]['Audio_File_Name']
        audio_path = os.path.join(self.audio_dir, f"{audio_name}.flac")

        waveform, sample_rate = torchaudio.load(audio_path)

        # --- THE FIX: Pad or Truncate to ensure uniform length ---
        num_channels, num_frames = waveform.shape
        if num_frames > self.max_length:
            # Truncate the audio if it's too long
            waveform = waveform[:, :self.max_length]
        elif num_frames < self.max_length:
            # Pad with silence (zeros) if it's too short
            pad_amount = self.max_length - num_frames
            waveform = F.pad(waveform, (0, pad_amount))
        # ---------------------------------------------------------

        label_str = self.data.iloc[idx]['Label']
        label = 0 if label_str == 'bonafide' else 1
        label_tensor = torch.tensor(label, dtype=torch.float32)

        if self.transform:
            waveform = self.transform(waveform)

        return waveform, label_tensor

# Define the Mel-Spectrogram transform
mel_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=16000,
    n_mels=128,
    n_fft=1024,
    hop_length=512
)

# Initialize the updated Dataset
protocol_path = '/content/asvspoof_data/LA/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt'
audio_folder = '/content/asvspoof_data/LA/LA/ASVspoof2019_LA_train/flac'

train_dataset = ASVSpoofDataset(protocol_file=protocol_path, audio_dir=audio_folder, transform=mel_transform)

# Create the DataLoader for batching
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)

print(f"Total training samples: {len(train_dataset)}")

Total training samples: 25380


In [ ]:
import torch
import torch.nn as nn

class EchoAuthentic(nn.Module):
    def __init__(self):
        super(EchoAuthentic, self).__init__()

        # 1. Spatial Feature Extraction (CNN)
        # Input shape expected: (Batch, Channels=1, Mels=128, Time)
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        # 2. Sequential Modeling (BiLSTM)
        # Our 128 Mel bands were pooled twice (128 -> 64 -> 32).
        # Feature dimension per time step = 32 channels * 32 mels = 1024
        self.lstm = nn.LSTM(
            input_size=1024,
            hidden_size=64,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        # 3. Attention Mechanism
        # BiLSTM outputs 128 features per time step (64 forward + 64 backward)
        self.attention = nn.Sequential(
            nn.Linear(128, 64),
            nn.Tanh(),
            nn.Linear(64, 1)
        )

        # 4. Final Classification
        self.classifier = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        # -- Pass through CNN --
        x = self.cnn(x)

        # -- Prepare for LSTM --
        # Current shape: (Batch, Channels, Features, Time)
        # We need: (Batch, Time, Channels * Features)
        batch_size, channels, features, time_steps = x.size()
        x = x.permute(0, 3, 1, 2).contiguous()
        x = x.view(batch_size, time_steps, channels * features)

        # -- Pass through BiLSTM --
        lstm_out, _ = self.lstm(x) # lstm_out shape: (Batch, Time, 128)

        # -- Apply Attention --
        attn_weights = self.attention(lstm_out) # Shape: (Batch, Time, 1)
        attn_weights = torch.softmax(attn_weights, dim=1)

        # Multiply weights by LSTM output and sum across the time dimension
        context_vector = torch.sum(attn_weights * lstm_out, dim=1) # Shape: (Batch, 128)

        # -- Final Prediction --
        # Output probability (closer to 0 = bonafide, closer to 1 = spoof)
        out = self.classifier(context_vector)
        return out.squeeze(-1)

# Instantiate the model and print the parameter count
model = EchoAuthentic()
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"EchoAuthentic Model initialized with {total_params:,} trainable parameters.")

EchoAuthentic Model initialized with 579,618 trainable parameters.


In [ ]:
import torch.optim as optim
from tqdm.auto import tqdm # For a clean progress bar

# 1. Hyperparameters & Device Setup
epochs = 10
learning_rate = 0.001
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training initialized on: {device}")

# Move model to GPU (Crucial for Colab)
model = EchoAuthentic().to(device)

# 2. Loss and Optimizer
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# 3. Initialize Weights & Biases Tracking
# This will prompt you to paste your W&B API key the first time you run it
wandb.init(project="EchoAuthentic", name="CNN-BiLSTM-Attn-BaseRun")

# 4. The Training Loop
for epoch in range(epochs):
    model.train() # Set model to training mode
    running_loss = 0.0

    # Progress bar for the batch loader
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")

    for waveforms, labels in progress_bar:
        # Move data to the GPU
        waveforms, labels = waveforms.to(device), labels.to(device)

        # Step 1: Clear old gradients
        optimizer.zero_grad()

        # Step 2: Forward pass
        outputs = model(waveforms)

        # Step 3: Calculate loss
        loss = criterion(outputs, labels)

        # Step 4: Backward pass (calculate gradients)
        loss.backward()

        # Step 5: Update weights
        optimizer.step()

        running_loss += loss.item()

        # Update progress bar UI
        progress_bar.set_postfix({'loss': f"{loss.item():.4f}"})

    # Calculate average loss for the epoch
    epoch_loss = running_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{epochs}] Completed - Average Loss: {epoch_loss:.4f}")

    # Log metrics to your W&B dashboard
    wandb.log({"epoch": epoch + 1, "train_loss": epoch_loss})

# Close W&B run when finished
wandb.finish()

Training initialized on: cuda


Epoch 1/10:   0%|          | 0/794 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7aa760109120>
Traceback (most recent call last):
Exception ignored in:   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
<function _MultiProcessingDataLoaderIter.__del__ at 0x7aa760109120>
    Traceback (most recent call last):
if w.is_alive():  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()

  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
       if w.is_alive(): 
      ^^ ^ ^ ^ ^^^^^^^^^^^^^^^
^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    ^^assert self._parent_pid == os.getpid(), 'can only test a child process'
^ 
   File "/usr/lib/pyth

Epoch [1/10] Completed - Average Loss: 0.2289


Epoch 2/10:   0%|          | 0/794 [00:00<?, ?it/s]

...

Epoch [9/10] Completed - Average Loss: 0.0227


Epoch 10/10:   0%|          | 0/794 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7aa760109120>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
Exception ignored in:     <function _MultiProcessingDataLoaderIter.__del__ at 0x7aa760109120>if w.is_alive():

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
      self._shutdown_workers()
   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
     if w.is_alive(): 
         ^^^^^^^^^^^^^^^^^^
^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'^Exception ignored in: ^<funct

Epoch [10/10] Completed - Average Loss: 0.0185


epoch,▁▂▃▃▄▅▆▆▇█
train_loss,█▄▃▂▂▂▁▁▁▁
epoch,10
train_loss,0.01854


In [ ]:
import torch
import os

# 1. Define filenames
model_filename = "echo_authentic_v1.pth"
colab_save_path = f"/content/{model_filename}"
drive_save_path = f"/content/drive/MyDrive/EchoAuthentic/models/{model_filename}"

# 2. Ensure the directory exists in your Google Drive
os.makedirs("/content/drive/MyDrive/EchoAuthentic/models", exist_ok=True)

# 3. Save the state dictionary (the weights) locally in Colab
torch.save(model.state_dict(), colab_save_path)
print(f"✅ Saved weights locally to: {colab_save_path}")

# 4. Copy it over to your permanent Google Drive
torch.save(model.state_dict(), drive_save_path)
print(f"🚀 Permanently backed up weights to Google Drive: {drive_save_path}")

✅ Saved weights locally to: /content/echo_authentic_v1.pth
🚀 Permanently backed up weights to Google Drive: /content/drive/MyDrive/EchoAuthentic/models/echo_authentic_v1.pth


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, roc_curve
import torch
from tqdm.auto import tqdm

# 1. Load the Evaluation (Dev) Dataset
# FIXED PATH: Notice the '.trl.txt' extension at the end
protocol_path_dev = '/content/asvspoof_data/LA/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.dev.trl.txt'
audio_folder_dev = '/content/asvspoof_data/LA/LA/ASVspoof2019_LA_dev/flac'

# We reuse the exact same ASVSpoofDataset class and transform from Phase 1
dev_dataset = ASVSpoofDataset(protocol_file=protocol_path_dev, audio_dir=audio_folder_dev, transform=mel_transform)
dev_loader = DataLoader(dev_dataset, batch_size=32, shuffle=False, num_workers=2)

print(f"Total evaluation samples: {len(dev_dataset)}")

# 2. Evaluation Loop
model.eval() # Turn off dropout layers for deterministic testing
all_preds = []
all_labels = []

print("Running inference on unseen data...")
with torch.no_grad(): # Disable gradient memory to massively speed up inference
    for waveforms, labels in tqdm(dev_loader, desc="Evaluating"):
        waveforms = waveforms.to(device)

        # Get raw probabilities from the model
        outputs = model(waveforms)

        # Store predictions and true labels
        all_preds.extend(outputs.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Convert to numpy arrays for sklearn processing
all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# 3. Calculate Metrics
# Calculate standard Accuracy (using 0.5 as the default binary threshold)
binary_preds = (all_preds >= 0.5).astype(int)
accuracy = accuracy_score(all_labels, binary_preds)

# Calculate Equal Error Rate (EER) using Sklearn's ROC function
fpr, tpr, thresholds = roc_curve(all_labels, all_preds)
fnr = 1 - tpr

# The EER is the point where the difference between FPR and FNR is closest to zero
eer_index = np.nanargmin(np.absolute(fnr - fpr))
eer = fpr[eer_index]
optimal_threshold = thresholds[eer_index]

print("\n=== Final Evaluation Metrics ===")
print(f"Accuracy: {accuracy * 100:.2f}%")
print(f"Equal Error Rate (EER): {eer * 100:.2f}%")
print(f"Optimal Threshold for Deployment: {optimal_threshold:.4f}")

# 4. Log final metrics to W&B
wandb.init(project="EchoAuthentic", resume=True) # Reconnect to your active run
wandb.log({"val_accuracy": accuracy, "val_eer": eer, "optimal_threshold": optimal_threshold})
wandb.finish()

Total evaluation samples: 24844
Running inference on unseen data...


Evaluating:   0%|          | 0/777 [00:00<?, ?it/s]


=== Final Evaluation Metrics ===
Accuracy: 96.44%
Equal Error Rate (EER): 6.51%
Optimal Threshold for Deployment: 0.9855


optimal_threshold,▁
val_accuracy,▁
val_eer,▁
optimal_threshold,0.98549
val_accuracy,0.96438
val_eer,0.06515
